# 07 - Multi-View Dataset Review

Review only, no training. Loads the existing State Farm metadata, looks at the single-view limitation and the 10-class taxonomy, and explains why cross-view evaluation is needed. Also creates empty placeholder tables in the unified schema for the next dataset, and saves reference tables to `results/tables/`.

Schema described in the "unified multi-view schema" section of the root `README.md` / `data/README.md`.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pd.set_option("display.max_columns", None)

## 1. Load existing State Farm metadata

Reads the already-produced metadata file, doesn't modify it.

In [2]:
state_farm_metadata_path = PROCESSED_DIR / "state_farm_clean_metadata.csv"
state_farm_df = pd.read_csv(state_farm_metadata_path)

print("State Farm metadata shape:", state_farm_df.shape)
print("Columns:", state_farm_df.columns.tolist())
print("Number of unique subjects:", state_farm_df["subject"].nunique())

state_farm_df.head()

State Farm metadata shape: (22424, 5)
Columns: ['subject', 'classname', 'img', 'filepath', 'file_exists']
Number of unique subjects: 26


,subject,classname,img,filepath,file_exists
0,p002,c0,img_44733.jpg,D:\MSC_PROJECT\state-farm-distracted-driver-de...,True
1,p002,c0,img_72999.jpg,D:\MSC_PROJECT\state-farm-distracted-driver-de...,True
2,p002,c0,img_25094.jpg,D:\MSC_PROJECT\state-farm-distracted-driver-de...,True
3,p002,c0,img_69092.jpg,D:\MSC_PROJECT\state-farm-distracted-driver-de...,True
4,p002,c0,img_92629.jpg,D:\MSC_PROJECT\state-farm-distracted-driver-de...,True


## 2. Current State Farm camera-view limitation

State Farm images are all from one fixed side-mounted camera — no `camera_view` column since there was never more than one view. This is the core limitation motivating the multi-view stage: a model only trained/validated on this view has no evidence it'll work if the camera moves (dashboard, mirror, front-facing).

In [3]:
print("State Farm has exactly ONE camera view for every image in this dataset.")
print("Camera view label used in the unified schema for all State Farm rows: 'side'")

State Farm has exactly ONE camera view for every image in this dataset.
Camera view label used in the unified schema for all State Farm rows: 'side'


## 3. Current 10-class State Farm taxonomy

New datasets in this stage need to map onto these same 10 classes (see the unified schema section in `README.md` / `data/README.md` for mapping rules, including the driver's-own-left/right rule).

In [4]:
state_farm_classes = pd.DataFrame({
    "label_id": list(range(10)),
    "class_code": [f"c{i}" for i in range(10)],
    "label_name": [
        "safe_driving",
        "texting_right",
        "phone_right",
        "texting_left",
        "phone_left",
        "adjusting_radio",
        "drinking",
        "reaching_behind",
        "hair_or_makeup",
        "talking_to_passenger",
    ],
})

display(state_farm_classes)

class_distribution = state_farm_df["classname"].value_counts().sort_index()
display(class_distribution)

,label_id,class_code,label_name
0,0,c0,safe_driving
1,1,c1,texting_right
2,2,c2,phone_right
3,3,c3,texting_left
4,4,c4,phone_left
5,5,c5,adjusting_radio
6,6,c6,drinking
7,7,c7,reaching_behind
8,8,c8,hair_or_makeup
9,9,c9,talking_to_passenger


classname
c0    2489
c1    2267
c2    2317
c3    2346
c4    2326
c5    2312
c6    2325
c7    2002
c8    1911
c9    2129
Name: count, dtype: int64

## 4. Why cross-view evaluation is needed

- A model can look accurate (84.67% test accuracy for MobileNetV2) on its training view while failing on an unseen camera position — hand positions, occlusion, and left/right all change with viewpoint.
- Left/right classes are fragile: a front camera mirrors the image vs a side camera, so naive relabeling would swap `texting_left`/`texting_right`.
- Real deployment (and the FPGA phase) may use a different camera position than State Farm's, so this is a practical requirement, not just academic.
- Needs real multi-view data — candidates listed in the summary table below (AUC V2 recommended first).

## 5. Placeholder unified-schema table

Empty table using the unified schema (columns: image_path, label_id, label_name, subject_id, dataset_name, camera_view, vehicle_id, lighting, modality, split, is_synthetic, notes — see `README.md` / `data/README.md`). Zero rows for now — once a new dataset (e.g. AUC V2) is approved and downloaded, its adapter appends rows in this shape.

In [5]:
unified_schema_columns = [
    "image_path",
    "label_id",
    "label_name",
    "subject_id",
    "dataset_name",
    "camera_view",
    "vehicle_id",
    "lighting",
    "modality",
    "split",
    "is_synthetic",
    "notes",
]

unified_placeholder_df = pd.DataFrame(columns=unified_schema_columns)
display(unified_placeholder_df)

placeholder_path = TABLES_DIR / "unified_multiview_metadata_template.csv"
unified_placeholder_df.to_csv(placeholder_path, index=False)
print("Saved empty unified-schema template to:", placeholder_path)

,image_path,label_id,label_name,subject_id,dataset_name,camera_view,vehicle_id,lighting,modality,split,is_synthetic,notes


Saved empty unified-schema template to: D:\MSC_PROJECT\results\tables\unified_multiview_metadata_template.csv


## 6. Known camera views per candidate dataset (reference table)

Saved as a reference table for later notebooks (e.g. notebook 08's builder).

In [6]:
dataset_view_summary = pd.DataFrame([
    {"dataset_name": "state_farm", "camera_views": "side", "status": "in use (baseline)"},
    {"dataset_name": "auc_v2", "camera_views": "front (passenger-seat facing driver)", "status": "recommended next - not yet requested"},
    {"dataset_name": "100_driver", "camera_views": "front_left, front, front_right, side_right", "status": "candidate - not yet requested"},
    {"dataset_name": "drive_and_act", "camera_views": "6 views (face, body, hands)", "status": "candidate - later phase"},
    {"dataset_name": "sam_dd", "camera_views": "front, side", "status": "candidate - verify class list first"},
    {"dataset_name": "dmd", "camera_views": "face, body, hands (RGB only post-2026 revision)", "status": "low priority for this task"},
])

display(dataset_view_summary)

summary_path = TABLES_DIR / "multiview_dataset_camera_view_summary.csv"
dataset_view_summary.to_csv(summary_path, index=False)
print("Saved camera-view summary to:", summary_path)

,dataset_name,camera_views,status
0,state_farm,side,in use (baseline)
1,auc_v2,front (passenger-seat facing driver),recommended next - not yet requested
2,100_driver,"front_left, front, front_right, side_right",candidate - not yet requested
3,drive_and_act,"6 views (face, body, hands)",candidate - later phase
4,sam_dd,"front, side",candidate - verify class list first
5,dmd,"face, body, hands (RGB only post-2026 revision)",low priority for this task


Saved camera-view summary to: D:\MSC_PROJECT\results\tables\multiview_dataset_camera_view_summary.csv


## Summary

Didn't train or download anything. Confirmed State Farm's single-view limitation, restated the 10-class taxonomy, and prepared empty template tables for the next dataset.

Next: `08_unified_multiview_dataset_builder.ipynb` — the actual per-dataset adapter code that fills in the placeholder table.